[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syaikhipin/kdd26-memdiag/blob/tutorial-rebuild/experiment/github_submission/tutorial/phase1_memory_architectures/04_retrieval_multi_agent.ipynb)

> **Run this notebook in Google Colab** — click the badge above. The setup cell auto-clones the repo.


# Phase 1D - Retrieval & multi-agent (20-23)

**Phase 1D - Retrieval & multi-agent (20-23)** - an independent notebook (runnable standalone in Colab or locally).

Covers: Retrieval & multi-agent (20-23) - find and share memories.

Attribution: adapted from *Agent Memory Techniques* by Nir Diamant (https://github.com/NirDiamant/Agent_Memory_Techniques), Apache-2.0. Inline demos are original.

## 0. Setup (self-contained - run this first)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "tutorial-rebuild", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "experiment" / "github_submission" / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

from embedder import TfidfHashEmbedder
from llm_client import LLMConfig, make_client, OfflineLLMClient
from memory_core import MemoryRecord
from providers import build_provider
emb = TfidfHashEmbedder()
try:
    llm = make_client(LLMConfig(backend='openai-compatible' if os.environ.get('OPENAI_API_KEY') else 'offline',
                                base_url=os.environ.get('OPENAI_BASE_URL','https://api.openai.com/v1'),
                                model='gpt-4o'))
except Exception:
    llm = OfflineLLMClient()
print('LLM backend:', llm.backend, '| API key:', bool(os.environ.get('OPENAI_API_KEY')))
TURNS = [
 ("Alice", "Hi, I am Alice. I work as a data scientist at a health-tech startup in Berlin."),
 ("Bob", "I am Bob, an ML engineer in Athens. I prefer PyTorch."),
 ("Alice", "We deploy on Kubernetes and track runs with Weights and Biases."),
 ("Bob", "Our training run failed last night with CUDA OOM at batch 256."),
 ("Alice", "We hit that before. Reducing batch to 64 and enabling gradient checkpointing fixed it."),
 ("Bob", "Our best val_loss was 0.423 with lr=3e-4 and weight_decay=0.01."),
 ("Alice", "I live in Prenzlauer Berg. My favorite coffee shop is on Kollwitzplatz."),
 ("Bob", "Let us sync next Tuesday at 10am CET."),
]
records = [MemoryRecord(f"t{i}", f"{w}: {t}", {"session_id": "s1" if i < 4 else "s2"}) for i, (w, t) in enumerate(TURNS)]
print("setup OK | turns =", len(TURNS))


## Retrieval & multi-agent (20-23) - find and share memories

**20 - Retrieval Patterns** · Compare semantic/recency/hybrid scoring

In [ ]:
# Technique 20 - Retrieval Patterns: Compare semantic/recency/hybrid scoring
vecs=[emb.embed(t) for _,t in TURNS]; qv=emb.embed('Where does Alice live?')
sem=max(range(len(TURNS)), key=lambda i: float(vecs[i]@qv))
print('20 semantic top:', TURNS[sem][0], '| recency top:', TURNS[-1][0])

**21 - Cross-Session** · Save/reload agent state across sessions

In [ ]:
# Technique 21 - Cross-Session: Save/reload agent state across sessions
import json
snap=json.dumps({'last_topic':'OOM fix'})
print('21 cross-session snapshot:', snap)

**22 - Multi-Agent Shared** · Shared stores + message passing

In [ ]:
# Technique 22 - Multi-Agent Shared: Shared stores + message passing
shared={}
shared['alice']=['data scientist']; shared['bob']=shared.get('bob',[])+['OOM fix']
print('22 shared store:', shared)

**23 - Memory with Tools** · save/search/forget as callable tools

In [ ]:
# Technique 23 - Memory with Tools: save/search/forget as callable tools
store=[]
def mem_save(x): store.append(x); return 'saved'
def mem_search(q): return store[-1] if store else None
print('23 tools:', mem_save('OOM fix'), mem_search('x'))